# October Task #1: T-learner (separate treatment and control models)

Train one visit model on **treatment** rows and one on **control** rows (training split only,
features `f0` to `f11` only). Score every record with both. The uplift score is

`predicted visit probability if treated - predicted visit probability if control`.

Rules followed here: no `exposure`, no test data (the test file is opened once, in November),
and scoring/metrics come from the shared `src/uplift_eval.py` so every model is measured the same way.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.uplift_eval import (
    FEATURE_COLUMNS, SPECIAL_COLUMNS, area_above_random, cumulative_uplift,
    overall_itt, percent_view, plot_cumulative, plot_uplift_by_group,
    random_ranking_null, summarize_null, uplift_by_group,
)

RANDOM_SEED = 2026
EXPECTED_COLUMNS = FEATURE_COLUMNS + SPECIAL_COLUMNS

In [ ]:
# Train and validation only. The test file is intentionally not loaded here.
train_df = pd.read_csv(ROOT / "data" / "train.csv.gz")
val_df = pd.read_csv(ROOT / "data" / "validation.csv.gz")

print(train_df.shape, val_df.shape)
assert set(train_df.columns) == set(EXPECTED_COLUMNS), "columns don't match what the notebook expects"
assert set(val_df.columns) == set(EXPECTED_COLUMNS), "columns don't match what the notebook expects"

In [ ]:
train_treat = train_df[train_df["treatment"] == 1]
train_ctrl = train_df[train_df["treatment"] == 0]

In [ ]:
for name, g in [("treatment", train_treat), ("control", train_ctrl)]:
    print(f"{name}: {len(g):,} rows, {g['visit'].sum():,} visits, visit rate {g['visit'].mean():.4f}")

## Fit the two models

Both are regularized logistic regressions. The penalty strength `C` is chosen with 5-fold
cross-validation **inside each training group**, so validation is never used to tune it.

Note: the two models can choose different `C` values. The control group is much smaller
(about 105k rows and about 4,000 visits, versus about 595k rows for treatment), so it is the noisier model.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def fit_regularized_logreg(df):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(Cs=[0.001, 0.01, 0.1, 1, 10], cv=cv,
                             scoring="neg_log_loss", max_iter=1000),
    )
    return model.fit(df[FEATURE_COLUMNS], df["visit"])

tl_treat = fit_regularized_logreg(train_treat)
tl_ctrl = fit_regularized_logreg(train_ctrl)

print("chosen C (treatment model):", tl_treat[-1].C_[0])
print("chosen C (control model):  ", tl_ctrl[-1].C_[0])

## Score validation

Every validation row gets both predictions. Only validation is scored, because test data is
reserved for the final evaluation in November.

In [ ]:
p_t = tl_treat.predict_proba(val_df[FEATURE_COLUMNS])[:, 1]
p_c = tl_ctrl.predict_proba(val_df[FEATURE_COLUMNS])[:, 1]
val_scores = pd.DataFrame({"p_treat": p_t, "p_ctrl": p_c, "uplift": p_t - p_c}, index=val_df.index)
val_scores.describe()

## Calibration check (validation)

Are the predicted probabilities close to what really happened? Most records have very small
probabilities, so the plot mostly tells us about the two ends of the range.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

val_treat = val_df[val_df["treatment"] == 1]
val_ctrl = val_df[val_df["treatment"] == 0]

checks = [
    ("Treatment model", val_treat, val_scores.loc[val_treat.index, "p_treat"], train_treat["visit"].mean()),
    ("Control model", val_ctrl, val_scores.loc[val_ctrl.index, "p_ctrl"], train_ctrl["visit"].mean()),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, group, preds, train_rate) in zip(axes, checks):
    y = group["visit"]
    frac_pos, mean_pred = calibration_curve(y, preds, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label="model")
    lim = max(mean_pred.max(), frac_pos.max()) * 1.1
    ax.plot([0, lim], [0, lim], "k--", label="perfect calibration")
    ax.set_xlabel("Mean predicted visit probability")
    ax.set_ylabel("Observed visit rate")
    ax.set_title(name)
    ax.legend()

    brier = brier_score_loss(y, preds)
    brier_const = brier_score_loss(y, np.full(len(y), train_rate))
    print(f"{name}: n={len(y):,}, mean predicted={preds.mean():.4f}, observed={y.mean():.4f}, "
          f"Brier={brier:.5f} (constant-rate baseline {brier_const:.5f})")
plt.tight_layout()
plt.show()

## Uplift by group on validation (shared evaluation code)

Group 1 = highest uplift score. Uplift = treatment visit rate minus control visit rate inside the group,
with a 95% CI. Incremental visits = group size x uplift.

In [ ]:
t_val = val_df["treatment"].to_numpy()
y_val = val_df["visit"].to_numpy()
scores_val = val_scores["uplift"].to_numpy()

itt = overall_itt(t_val, y_val)
t_learner_table = uplift_by_group(scores_val, t_val, y_val)
t_learner_curve = cumulative_uplift(scores_val, t_val, y_val)

print(f"Overall validation ITT: {itt['uplift'] * 100:.3f} pp "
      f"-> {itt['incremental_visits']:.0f} incremental visits in {itt['n']:,} records")
display(percent_view(t_learner_table))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_uplift_by_group(t_learner_table, itt=itt, title="T-learner: uplift by group (validation)", ax=axes[0])
plot_cumulative({"T-learner": t_learner_curve}, title="T-learner: cumulative incremental visits", ax=axes[1])
plt.tight_layout()
plt.show()

### How to read this

- The top group is the only one clearly above zero. Most other groups sit close to zero with
  confidence intervals that include zero.
- **Group 10 (lowest score) is not "the ad hurts these people."** It contains frequent visitors, where
  subtracting two noisy probability models can produce extreme scores. Its uplift is slightly positive
  with a wide interval.
- Group-by-group incremental visits do not add up exactly to the overall total, because each is a small,
  noisy estimate. Treat the totals as approximate, not exact.

## How much could random chance explain?

Score the same validation data with 200 random rankings. A real model should beat this range, not just zero.

In [ ]:
null_df = random_ranking_null(t_val, y_val, n_seeds=200, top_fraction=0.3)
null_summary = summarize_null(null_df)
display(null_summary.round(4))

row = t_learner_curve.iloc[(t_learner_curve["fraction_targeted"] - 0.3).abs().argmin()]
print(f"T-learner, top 30% of ranking: {row['incremental_visits']:.0f} incremental visits "
      f"(95% CI {row['incremental_ci_low']:.0f} to {row['incremental_ci_high']:.0f})")
print(f"Random ranking, top 30%: 97.5th percentile = {null_summary.loc['incremental_top_30pct', 'p97.5']:.0f}")
print(f"T-learner area above random: {area_above_random(t_learner_curve):.3f} "
      f"(random 97.5th percentile = {null_summary.loc['area_above_random', 'p97.5']:.3f})")

## Save validation scores for the team comparison (Task #3)

One array, same row order as `validation.csv.gz`. The comparison notebook loads this file.

In [ ]:
SCORE_DIR = ROOT / "scores"
SCORE_DIR.mkdir(exist_ok=True)
np.save(SCORE_DIR / "val_t_learner.npy", scores_val)

saved = np.load(SCORE_DIR / "val_t_learner.npy")
assert saved.shape == (len(val_df),) and np.isfinite(saved).all()
print("Saved", SCORE_DIR / "val_t_learner.npy", saved.shape)